# 06 - Transformer / Self-Attention
## Clasificación de Marcha Patológica

**Autor:** Weimar Andres Arenas Gonzalez  
**Curso:** Fundamentos de Deep Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WeimarArenas/ProyectoDL20261/blob/main/06%20-%20Transformer_SelfAttention.ipynb)

---
**Objetivo:** Implementar un clasificador basado en Transformer (Self-Attention) para secuencias de esqueleto.

**Hipótesis:** El mecanismo de self-attention puede ponderar simultáneamente todos los 50 frames
sin la limitación secuencial de RNN/LSTM/GRU, potencialmente capturando dependencias
de largo alcance en la secuencia de marcha (p.ej., correlación entre frame 5 y frame 45).

**Referencia:** Vaswani et al. (2017) — 'Attention Is All You Need'  
**Objetivo de rendimiento:** Superar el 93.67% del GRU (Jun et al., 2020)

In [ ]:
import subprocess, sys
pkgs = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'tensorflow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pickle
from collections import Counter
from sklearn.model_selection import LeaveOneGroupOut

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')


## 1. Configuración del entorno

In [ ]:
# ============================================================
# FUNCIONES DE CARGA — Formato real del dataset GIST
# CSV: tab-separado, sin header, 101 columnas
# Col 0: timestamp | cols (1+j*4)+1,+2,+3 = x,y,z para joint j
# Directorio padre: human{N}_{gait_type}{rep}/
# ============================================================

import re as _re

# Indices de features: x,y,z de cada articulacion (omite timestamp y joint_id)
FEATURE_COLS = []
for _j in range(25):
    _base = 1 + _j * 4
    FEATURE_COLS.extend([_base + 1, _base + 2, _base + 3])
# Resultado: [2,3,4, 6,7,8, ..., 98,99,100]  -> 75 valores por fila

_GAIT_PAT = _re.compile(
    r'human(\d+)_(normal|antalgic|lurch|steppage|stiff_legged|trendelenburg)\d+'
)
CLASS_MAP = {
    'normal': 0, 'antalgic': 1, 'stiff_legged': 2,
    'lurch': 3, 'steppage': 4, 'trendelenburg': 5,
}
CLASS_NAMES = ['Normal', 'Antalgic', 'Stiff-legged', 'Lurching', 'Steppage', 'Trendelenburg']


def extract_label_and_subject(filepath):
    """Extrae clase y sujeto del nombre del directorio padre del CSV."""
    dirname = os.path.basename(os.path.dirname(filepath))
    m = _GAIT_PAT.match(dirname)
    if not m:
        return None, None
    return CLASS_MAP.get(m.group(2)), int(m.group(1))


def load_csv_sequence(filepath, skip_frames=10, n_frames=50):
    """
    Lee un CSV del GIST dataset (tab-separado, 101 cols, sin header).
    Retorna array (n_frames, 75) omitiendo los primeros skip_frames.
    """
    try:
        df = pd.read_csv(filepath, header=None, sep='\t')
        if df.shape[1] != 101 or len(df) < skip_frames + n_frames:
            return None
        seq = df.iloc[skip_frames:skip_frames + n_frames, FEATURE_COLS].values.astype(np.float32)
        return None if np.any(np.isnan(seq)) else seq
    except Exception:
        return None


def load_full_dataset(data_dir, verbose=True):
    """
    Carga todos los CSV validos del GIST dataset.
    data_dir: ruta a Pathological_Gaits/
    Retorna: X (N,50,75), y (N,), S (N,)
    """
    sequences, labels, subjects = [], [], []
    skipped = 0
    for dirname in sorted(os.listdir(data_dir)):
        dirpath = os.path.join(data_dir, dirname)
        if not os.path.isdir(dirpath):
            continue
        for fname in sorted(os.listdir(dirpath)):
            if not fname.endswith('.csv'):
                continue
            fpath = os.path.join(dirpath, fname)
            label, subject = extract_label_and_subject(fpath)
            if label is None:
                skipped += 1
                continue
            seq = load_csv_sequence(fpath)
            if seq is None:
                skipped += 1
                continue
            sequences.append(seq)
            labels.append(label)
            subjects.append(subject)
    if verbose:
        print(f'Secuencias cargadas: {len(sequences)} | Descartadas: {skipped}')
    return (np.array(sequences, dtype=np.float32),
            np.array(labels, dtype=np.int32),
            np.array(subjects, dtype=np.int32))


In [ ]:
# ---- Cargar datos (desde .npy cache o desde CSV) ----------------------------
if not os.path.exists('pathological_gait_datasets'):
    os.system('git clone --quiet https://github.com/kooksung/pathological_gait_datasets.git')

DATA_DIR = 'pathological_gait_datasets/Pathological_Gaits'

if os.path.exists('X_all.npy') and os.path.exists('y_labels.npy'):
    X    = np.load('X_all.npy')
    y    = np.load('y_labels.npy')
    S    = np.load('S_subjects.npy')
    print(f'Datos cargados desde .npy: X={X.shape}')
else:
    assert os.path.isdir(DATA_DIR), f'No se encontro el dataset en: {DATA_DIR}'
    X, y, S = load_full_dataset(DATA_DIR)
    np.save('X_all.npy', X)
    np.save('y_labels.npy', y)
    np.save('S_subjects.npy', S)
    print(f'Datos cargados desde CSV y guardados: X={X.shape}')

# Seleccion de joints de piernas
LEG_JOINT_INDICES = [0, 1, 12, 13, 14, 15, 16, 17, 18, 19]
leg_feat = [f for j in LEG_JOINT_INDICES for f in [j*3, j*3+1, j*3+2]]
X_legs = X[:, :, leg_feat]

print(f'X={X.shape} | X_legs={X_legs.shape}')
print(f'Clases: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'Sujetos: {sorted(np.unique(S))}')


## 2. Bloques del Transformer

### Componentes principales:
1. **Positional Encoding:** Información de posición temporal (frame 0, 1, ..., 49)
2. **Multi-Head Self-Attention:** Pesos de atención entre todos los pares de frames
3. **Feed-Forward Network:** Capa densa aplicada independientemente a cada posición
4. **Layer Normalization + Residual:** Estabilidad del entrenamiento

In [ ]:
class PositionalEncoding(layers.Layer):
    """
    Codificación posicional aprendida (Vaswani et al., 2017).
    Para secuencias de marcha, las posiciones son los frames temporales (0-49).
    Se usa una capa Embedding aprendida en lugar de la versión sinusoidal fija,
    ya que la secuencia es corta (50 frames) y el embedding puede adaptarse.
    """
    def __init__(self, max_len, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)
        self.max_len = max_len
    
    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        return x + self.pos_emb(positions)
    
    def get_config(self):
        config = super().get_config()
        config.update({'max_len': self.max_len, 'embed_dim': self.pos_emb.output_dim})
        return config


class TransformerBlock(layers.Layer):
    """
    Bloque Transformer: Multi-Head Attention + FFN + LayerNorm + Residual.
    
    Arquitectura (Vaswani et al., 2017):
    Input → MultiHeadAttention → Add+Norm → FFN → Add+Norm → Output
    
    Parámetros:
        embed_dim:  dimensión de embedding (modelo)
        num_heads:  número de cabezas de atención
        ff_dim:     dimensión interna de la red feed-forward
        rate:       tasa de dropout
    """
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
        # Guardar config para serialización
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
    
    def call(self, inputs, training=None):
        # Self-attention (Q=K=V=inputs)
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)  # Residual + Norm
        
        # Feed-Forward Network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)  # Residual + Norm
    
    def get_config(self):
        config = super().get_config()
        config.update({'embed_dim': self.embed_dim, 'num_heads': self.num_heads,
                       'ff_dim': self.ff_dim, 'rate': self.rate})
        return config


print('Clases TransformerBlock y PositionalEncoding definidas.')

## 3. Arquitectura del clasificador Transformer

```
Input (50, 75)
  → Dense (proyección a embed_dim)
  → Positional Encoding
  → TransformerBlock × n_blocks
  → GlobalAveragePooling1D (agrega información de todos los frames)
  → Dense(128, ReLU) → Dropout
  → Softmax(6)
```

In [ ]:
def build_transformer_classifier(
    input_shape,
    n_classes=6,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    n_blocks=2,
    dropout_rate=0.1,
    l2_reg=1e-4
):
    """
    Clasificador Transformer para secuencias de esqueleto 3D.
    
    Parámetros:
        input_shape:  (T, F) — (50, 75) o (50, 30)
        embed_dim:    dimensión de embedding del Transformer
        num_heads:    cabezas de atención (debe dividir embed_dim)
        ff_dim:       dimensión de la FFN interna
        n_blocks:     número de bloques Transformer apilados
        dropout_rate: tasa de dropout en atención y FFN
    """
    T, F = input_shape
    reg = keras.regularizers.l2(l2_reg)
    
    inputs = layers.Input(shape=input_shape, name='input')
    
    # Proyección lineal: (T, F) → (T, embed_dim)
    x = layers.Dense(embed_dim, kernel_regularizer=reg, name='input_proj')(inputs)
    
    # Codificación posicional (frame 0 a T-1)
    x = PositionalEncoding(max_len=T, embed_dim=embed_dim, name='pos_enc')(x)
    x = layers.Dropout(dropout_rate)(x)
    
    # Bloques Transformer
    for i in range(n_blocks):
        x = TransformerBlock(
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            rate=dropout_rate,
            name=f'transformer_{i+1}'
        )(x)
    
    # Pooling global sobre la dimensión temporal
    # GlobalAveragePooling1D: promedia las representaciones de todos los frames
    x = layers.GlobalAveragePooling1D(name='global_avg_pool')(x)
    
    # Clasificador MLP
    x = layers.Dense(128, activation='relu', kernel_regularizer=reg, name='fc1')(x)
    x = layers.Dropout(dropout_rate * 2)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=reg, name='fc2')(x)
    x = layers.Dropout(dropout_rate)(x)
    
    outputs = layers.Dense(n_classes, activation='softmax', name='softmax')(x)
    
    return keras.Model(inputs, outputs, name='Transformer_Gait_Classifier')


# Mostrar arquitectura
transformer_demo = build_transformer_classifier(
    input_shape=(50, 75),
    embed_dim=64, num_heads=4, ff_dim=128, n_blocks=2
)
transformer_demo.summary()
print(f'\nParámetros totales: {transformer_demo.count_params():,}')

## 4. Análisis del mecanismo de atención

Visualización de qué frames presta atención el modelo antes de entrenar (inicialización aleatoria).

In [ ]:
# Visualizar las matrices de atención de un bloque Transformer
# Esto muestra cómo el modelo pondera los frames entre sí

def get_attention_weights(model, X_sample, block_name='transformer_1'):
    """
    Extrae los pesos de atención de un bloque Transformer.
    Retorna tensor de forma (num_heads, T, T).
    """
    # Crear submodelo hasta la capa de atención
    att_layer = model.get_layer(block_name).att
    
    # Obtener salida de la capa de proyección
    proj_out = keras.Model(
        inputs=model.input,
        outputs=model.get_layer('pos_enc').output
    )(X_sample, training=False)
    
    # Calcular pesos de atención
    _, attention_scores = att_layer(
        proj_out, proj_out, return_attention_scores=True
    )
    return attention_scores.numpy()


# Tomar una muestra de cada clase
X_sample_norm = X[:6].copy()
scaler_demo = StandardScaler()
T, F = 50, 75
X_sample_2d = scaler_demo.fit_transform(X_sample_norm.reshape(-1, F))
X_sample_3d = X_sample_2d.reshape(6, T, F).astype(np.float32)

try:
    att_weights = get_attention_weights(transformer_demo, X_sample_3d)
    print(f'Forma de pesos de atención: {att_weights.shape}')  # (batch, heads, T, T)
    
    # Visualizar mapa de atención (promedio de cabezas para la primera muestra)
    att_mean = att_weights[0].mean(axis=0)  # (T, T)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(att_mean, cmap='hot', aspect='auto')
    plt.colorbar(im, ax=ax, label='Peso de atención')
    ax.set_title('Mapa de Self-Attention (promedio de cabezas)\n'
                 'Frame i ↔ Frame j', fontsize=12, fontweight='bold')
    ax.set_xlabel('Frame (key/value)')
    ax.set_ylabel('Frame (query)')
    plt.tight_layout()
    plt.savefig('attention_map_init.png', dpi=100, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'No se pudo extraer atención (modelo no entrenado): {e}')
    print('Se visualizará después del entrenamiento.')

## 5. Funciones de entrenamiento LOSO-CV

In [ ]:
def evaluate_fold(model, X_test, y_test):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    sensitivity = np.zeros(N_CLASSES)
    specificity = np.zeros(N_CLASSES)
    precision = np.zeros(N_CLASSES)
    for cls in range(N_CLASSES):
        TP = cm[cls, cls]; FN = cm[cls, :].sum() - TP
        FP = cm[:, cls].sum() - TP; TN = cm.sum() - TP - FN - FP
        sensitivity[cls] = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity[cls] = TN / (TN + FP) if (TN + FP) > 0 else 0
        precision[cls]   = TP / (TP + FP) if (TP + FP) > 0 else 0
    return {'accuracy': acc, 'confusion_matrix': cm,
            'sensitivity': sensitivity, 'specificity': specificity, 'precision': precision}


def train_loso_cv(X_data, y, S, model_builder_fn, model_name,
                  n_epochs=500, batch_size=64, patience=40, lr=5e-4):
    """
    LOSO-CV para el Transformer.
    Se usa lr=5e-4 y patience=40 (Transformer necesita más épocas para converger).
    """
    logo = LeaveOneGroupOut()
    results = []
    print(f'\n=== LOSO-CV: {model_name} ===')
    
    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_data, y, groups=S)):
        test_subj = int(np.unique(S[test_idx])[0])
        N_tr, T, F = X_data[train_idx].shape
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_data[train_idx].reshape(-1, F)).reshape(N_tr, T, F)
        X_te = scaler.transform(X_data[test_idx].reshape(-1, F)).reshape(len(test_idx), T, F)
        y_tr_oh = to_categorical(y[train_idx], num_classes=N_CLASSES)
        
        model = model_builder_fn((T, F))
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        t0 = time.time()
        hist = model.fit(
            X_tr, y_tr_oh,
            epochs=n_epochs,
            batch_size=batch_size,
            validation_split=0.1,
            callbacks=[
                EarlyStopping('val_accuracy', patience=patience,
                              restore_best_weights=True, verbose=0),
                ReduceLROnPlateau('val_loss', factor=0.5, patience=20,
                                  min_lr=1e-6, verbose=0)
            ],
            verbose=0
        )
        
        metrics = evaluate_fold(model, X_te, y[test_idx])
        metrics.update({'fold': fold_idx, 'test_subject': test_subj,
                        'history': hist.history, 'epochs': len(hist.history['loss']),
                        'time': time.time() - t0,
                        'model': model})  # Guardar modelo para extracción de atención
        results.append(metrics)
        
        print(f"  Fold {fold_idx:2d} | S{test_subj:02d} | Acc={metrics['accuracy']*100:.2f}% | "
              f"Épocas={metrics['epochs']} | {metrics['time']:.0f}s")
    
    accs = [r['accuracy'] for r in results]
    print(f'  → Media: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%')
    return results

print('Funciones listas.')

## 6. Entrenamiento del Transformer

In [ ]:
# Transformer — Todos los joints
results_transformer_all = train_loso_cv(
    X_data=X, y=y, S=S,
    model_builder_fn=lambda s: build_transformer_classifier(
        s, embed_dim=64, num_heads=4, ff_dim=128, n_blocks=2
    ),
    model_name='Transformer (todos joints)',
    lr=5e-4, patience=40
)

In [ ]:
# Transformer — Solo piernas (mejor input según el paper)
results_transformer_legs = train_loso_cv(
    X_data=X_legs, y=y, S=S,
    model_builder_fn=lambda s: build_transformer_classifier(
        s, embed_dim=64, num_heads=4, ff_dim=128, n_blocks=2
    ),
    model_name='Transformer (solo piernas)',
    lr=5e-4, patience=40
)

## 7. Análisis de atención post-entrenamiento

In [ ]:
# Visualizar mapas de atención del modelo entrenado en el último fold
last_fold_model = results_transformer_all[-1].get('model')

if last_fold_model is not None:
    last_fold_idx = results_transformer_all[-1]['fold']
    test_subj = results_transformer_all[-1]['test_subject']
    
    # Preparar muestra del sujeto de test
    test_mask = S == test_subj
    X_test_subj = X[test_mask]
    y_test_subj = y[test_mask]
    
    # Normalizar con el scaler del fold (re-calcular para visualización)
    train_mask = S != test_subj
    sc = StandardScaler()
    sc.fit(X[train_mask].reshape(-1, 75))
    X_test_norm = sc.transform(X_test_subj.reshape(-1, 75)).reshape(len(X_test_subj), 50, 75)
    
    # Una muestra por clase
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()
    
    for cls_id in range(N_CLASSES):
        cls_mask = y_test_subj == cls_id
        if cls_mask.sum() == 0:
            continue
        
        sample = X_test_norm[cls_mask][0:1]  # (1, 50, 75)
        
        try:
            # Extraer pesos de atención
            att_weights = get_attention_weights(last_fold_model, sample.astype(np.float32))
            att_mean = att_weights[0].mean(axis=0)  # (T, T)
            
            im = axes[cls_id].imshow(att_mean, cmap='hot', aspect='auto', vmin=0)
            plt.colorbar(im, ax=axes[cls_id])
            axes[cls_id].set_title(f'Atención — {CLASS_NAMES[cls_id]}', fontweight='bold',
                                    color='green' if cls_id == 0 else 'red')
            axes[cls_id].set_xlabel('Frame (key)')
            axes[cls_id].set_ylabel('Frame (query)')
        except Exception:
            axes[cls_id].set_visible(False)
    
    plt.suptitle('Mapas de Self-Attention por clase de marcha (modelo entrenado)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('attention_maps_trained.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print('Modelo no disponible para extracción de atención.')

## 8. Resultados y comparación

In [ ]:
# Resumen de resultados del Transformer
transformer_results = {
    'Transformer (todos)':    results_transformer_all,
    'Transformer (piernas)':  results_transformer_legs,
}

print(f'\n{"Modelo":25s} {"Accuracy":>10} {"Sens.":>8} {"Espec.":>8} {"Prec.":>8}')
print('-' * 65)

for name, results in transformer_results.items():
    accs = [r['accuracy'] for r in results]
    sens = np.mean([r['sensitivity'] for r in results], axis=0).mean()
    spec = np.mean([r['specificity'] for r in results], axis=0).mean()
    prec = np.mean([r['precision']   for r in results], axis=0).mean()
    print(f'{name:25s} {np.mean(accs)*100:9.2f}% {sens*100:7.2f}% {spec*100:7.2f}% {prec*100:7.2f}%')

print('-' * 65)
print('Objetivo (GRU solo piernas, Jun 2020): 93.67%')

In [ ]:
# Curvas de entrenamiento del Transformer vs GRU
# (Para comparar convergencia)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for results, name, color in [
    (results_transformer_all, 'Transformer (todos)', 'purple'),
    (results_transformer_legs, 'Transformer (piernas)', 'darkorange'),
]:
    # Promediar curvas de entrenamiento sobre los folds
    max_epochs = max(len(r['history']['accuracy']) for r in results)
    
    all_train_acc = []
    all_val_acc = []
    for r in results:
        acc_pad = np.pad(r['history']['accuracy'], (0, max_epochs - len(r['history']['accuracy'])),
                         constant_values=np.nan)
        val_pad = np.pad(r['history']['val_accuracy'], (0, max_epochs - len(r['history']['val_accuracy'])),
                         constant_values=np.nan)
        all_train_acc.append(acc_pad)
        all_val_acc.append(val_pad)
    
    mean_train = np.nanmean(all_train_acc, axis=0) * 100
    mean_val   = np.nanmean(all_val_acc,   axis=0) * 100
    x = np.arange(1, len(mean_train) + 1)
    
    axes[0].plot(x, mean_train, label=f'{name} (train)', color=color, alpha=0.6)
    axes[1].plot(x, mean_val,   label=f'{name} (val)',   color=color)

for ax, title in zip(axes, ['Accuracy de Entrenamiento', 'Accuracy de Validación']):
    ax.set_title(f'Transformer — {title}\n(promedio 10 folds)', fontweight='bold')
    ax.set_xlabel('Época')
    ax.set_ylabel('Accuracy (%)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transformer_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de confusión del mejor Transformer
best_trans_name = max(transformer_results.keys(),
                      key=lambda k: np.mean([r['accuracy'] for r in transformer_results[k]]))
best_trans_results = transformer_results[best_trans_name]
best_trans_acc = np.mean([r['accuracy'] for r in best_trans_results])

cm_total = sum(r['confusion_matrix'] for r in best_trans_results)
cm_pct = cm_total.astype(float) / cm_total.sum(axis=1, keepdims=True) * 100
cls_labels = [CLASS_NAMES[i] for i in range(N_CLASSES)]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Purples',
            xticklabels=cls_labels, yticklabels=cls_labels, ax=ax,
            cbar_kws={'label': '%'})
ax.set_title(f'Matriz de confusión — {best_trans_name}\nAccuracy={best_trans_acc*100:.2f}%',
             fontweight='bold')
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('cm_transformer.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Guardar resultados
results_to_save = {}
for name, results in transformer_results.items():
    accs = [r['accuracy'] for r in results]
    results_to_save[name] = {
        'accuracies': accs,
        'mean_acc': float(np.mean(accs)),
        'std_acc': float(np.std(accs)),
        'sensitivity': np.mean([r['sensitivity'] for r in results], axis=0).tolist(),
        'specificity': np.mean([r['specificity'] for r in results], axis=0).tolist(),
        'precision':   np.mean([r['precision']   for r in results], axis=0).tolist(),
        'confusion_matrix': sum(r['confusion_matrix'] for r in results).tolist(),
        'epochs_mean': np.mean([r['epochs'] for r in results]),
    }

with open('results_transformer.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print('Resultados guardados en results_transformer.pkl')
print(f'\n→ Siguiente: Notebook 07 - Comparación de todos los modelos')